# PySpark 4: reading and writing data

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-04-reading-and-writing-data.ipynb)

Order data arrives as CSV from one system, JSON from another and Parquet from a third, sometimes with broken
rows and an extra column added mid-month. Your job writes a clean, partitioned copy that other jobs can read
quickly. Getting reads and writes right is most of what makes a pipeline reliable.

This lesson reads CSV, JSON, Parquet and ORC with explicit schemas and options, handles malformed records,
records which file each row came from, reads partitioned folders, combines schema versions, and writes data
with the right save mode, layout and file count. JDBC sources close the lesson.

Each section explains how something works, where it matters and what to watch out for, then runs small examples. Read each output, then the **What to notice** note under it. Run the cells in order.

**Prerequisites:** [PySpark 3: types, functions and nulls](pyspark-03-types-functions-and-nulls.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered) and `orders` (one row per order; order 12 lands just after midnight UTC): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F
import json

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints, so a cell can pick out the lines that matter."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()

In [ ]:
def land(folder, name, text):
    """Write one raw file into WORK_DIR/folder, the way a source system would deliver it."""
    path = os.path.join(WORK_DIR, folder)
    os.makedirs(path, exist_ok=True)
    with open(os.path.join(path, name), "w") as handle:
        handle.write(text)
    return path


def files_in(path, suffix=""):
    return sorted(name for _, _, names in os.walk(path) for name in names if name.endswith(suffix) and not name.startswith((".", "_")))

## How Spark reads files

Every read goes through `spark.read`, a `DataFrameReader`:

```python
spark.read.format("csv").schema(schema).option("header", "true").load(path)
spark.read.schema(schema).csv(path, header=True)           # the same, shorter
```

The path can be one file, a folder (every file inside is read), or a glob such as `raw/2026-01-*/`. Spark splits
large files into chunks of about 128 MB (`spark.sql.files.maxPartitionBytes`) and packs small files together,
so the number of partitions depends on total size, not on the number of files.

Always pass a **schema** for text formats. Without one, Spark either reads everything as strings (CSV) or scans
the data an extra time to guess types (`inferSchema`, JSON), and a guess can change from one delivery to the next.

**Where it matters**
- **Landing zones:** a job reads every file a source delivered for one day.
- **Contracts:** the schema in your code is the data contract with the producer.

**Watch out for**
- `inferSchema` reads the data twice and can pick different types for different deliveries.
- Spark lists every file in a folder before reading. Millions of tiny files make listing slower than reading.

## Read CSV

CSV has no types and many dialects, so be explicit: `header`, `sep`, `quote` and `escape`, `nullValue` (the text
that means null), and `dateFormat` / `timestampFormat` for non-ISO dates. A schema in DDL form is the easiest way
to give types.

**Where it matters**
- **Exports from operational systems and spreadsheets,** still the most common delivery format.
- **Vendor feeds** with their own delimiters and date formats.

**Watch out for**
- A comma inside a quoted field only works with the right `quote` and `escape` settings.
- Without `nullValue`, the text `NULL` or `-` is read as a string, not as a missing value.
- A header row is just text to Spark: names in the schema win, so keep their order in line with the file.

### Example

Read a semicolon-separated file with quoted fields, European dates and `-` for missing values.

In [ ]:
csv_dir = land("raw_csv", "customers.csv",
    "customer_id;email;signup;segment\n"
    '1;"ada@example.com";03/01/2026;retail\n'
    '2;"grace; hopper@example.org";15/02/2026;-\n'
    "3;linus@example.fi;01/03/2026;wholesale\n")

customers_csv = (
    spark.read
    .schema("customer_id int, email string, signup date, segment string")
    .option("header", True)
    .option("sep", ";")
    .option("quote", '"')
    .option("nullValue", "-")
    .option("dateFormat", "dd/MM/yyyy")
    .csv(csv_dir)
)
customers_csv.show(truncate=False)
customers_csv.printSchema()

**What to notice:** The semicolon inside Grace's quoted email did not split the row. `15/02/2026` became a real `date` because of
`dateFormat`, and the `-` segment became null because of `nullValue`. Without these options, the dates would be
null and the segment would be the string `-`.

## Read JSON

Spark reads **JSON Lines** by default: one complete JSON object per line, which splits cleanly across tasks.
A file that is one big JSON document or array needs `multiLine=True`, and is then read by a single task.
Nested objects become `struct` columns and lists become `array` columns ([lesson 8](pyspark-08-nested-and-semi-structured-data.ipynb)).

**Where it matters**
- **Event and API data:** clickstream, webhooks, application logs.
- **Exports from document databases** such as MongoDB.

**Watch out for**
- A pretty-printed JSON file read without `multiLine` becomes one corrupt record per line.
- Fields missing from some records become null; fields not in your schema are silently ignored.

### Example

Read the same records as JSON Lines and as one multi-line JSON array.

In [ ]:
records = [
    {"order_id": 1, "customer": {"id": 1, "tier": "gold"}, "skus": ["SKU-4001", "SKU-4007"]},
    {"order_id": 2, "customer": {"id": 2, "tier": "silver"}, "skus": []},
]
lines_dir = land("raw_json_lines", "orders.json", "\n".join(json.dumps(r) for r in records))
array_dir = land("raw_json_array", "orders.json", json.dumps(records, indent=2))

schema = "order_id int, customer struct<id:int, tier:string>, skus array<string>"
spark.read.schema(schema).json(lines_dir).show(truncate=False)
spark.read.schema(schema).option("multiLine", True).json(array_dir).show(truncate=False)
print("array file without multiLine:")
spark.read.schema("order_id int, _corrupt_record string").json(array_dir).show(4, truncate=40)

**What to notice:** Both correct reads give two rows with a nested `customer` struct and a `skus` array. Reading the pretty-printed
array without `multiLine` turns every line into a corrupt record with a null `order_id`: on its own, no line of
that file is complete JSON.

## Read and write Parquet and ORC

**Parquet** and **ORC** are columnar formats: values of one column are stored together, compressed, with the
schema and min/max statistics in each file. That gives three advantages over CSV and JSON: types are preserved,
a query reads only the columns it needs, and filters can skip whole blocks of rows. Parquet is the default in
most lakehouses and the storage format under Delta Lake and Apache Iceberg tables.

**Avro** is a row-based format common in Kafka pipelines; it needs the separate `spark-avro` package
(`format("avro")`).

**Where it matters**
- **Every analytical table** after the raw landing zone.
- **Interchange between engines:** Spark, DuckDB, Trino, pandas and warehouses all read Parquet.

**Watch out for**
- Parquet files are immutable; "updating" a row means rewriting files. Table formats manage that for you.
- Many small Parquet files lose most of the format's advantages (lesson 13).

### Example

Write `orders` as Parquet and as ORC, compare sizes with CSV, and confirm the types survive a round trip.

In [ ]:
for fmt in ["parquet", "orc", "csv"]:
    path = os.path.join(WORK_DIR, f"orders_{fmt}")
    orders.coalesce(1).write.mode("overwrite").format(fmt).option("header", True).save(path)
    size = sum(os.path.getsize(os.path.join(path, f)) for f in files_in(path))
    print(f"{fmt:<8} {size:>6} bytes")

print(spark.read.parquet(os.path.join(WORK_DIR, "orders_parquet")).dtypes)
print(spark.read.option("header", True).csv(os.path.join(WORK_DIR, "orders_csv")).dtypes)

**What to notice:** On twelve rows the columnar files are larger than CSV, because each file carries a schema and statistics; the
compression advantage appears with real volumes. What matters here is the second part: Parquet returned
`int`, `timestamp` and `decimal(12,2)` columns, while the CSV copy came back as all strings and would need a
schema again.

## Handle malformed records

The `mode` option decides what happens to a row that does not fit the schema:

| mode | bad row |
| --- | --- |
| `PERMISSIVE` (default) | kept; bad fields become null; with `columnNameOfCorruptRecord`, the raw text is kept too |
| `DROPMALFORMED` | dropped, silently |
| `FAILFAST` | the whole read fails |

The usual production choice is `PERMISSIVE` with a corrupt-record column, so bad rows can be counted and
quarantined instead of lost or fatal.

**Where it matters**
- **Vendor deliveries** where one bad row should not stop a day's load.
- **Strict contracts** where any bad row means the delivery must be rejected (`FAILFAST`).

**Watch out for**
- `DROPMALFORMED` hides data loss. Use it only when losing rows is acceptable and agreed.
- Spark refuses a query that selects only the corrupt-record column of an uncached read. Cache the parsed frame first.
- Always reconcile: rows landed = rows parsed + rows rejected.
- `count()` alone does not parse rows, so it cannot detect malformed records. Count after selecting real columns, or from a cached frame.

### Example

Read a JSON file with one bad amount in all three modes.

In [ ]:
bad_dir = land("raw_bad", "orders.json", "\n".join([
    '{"order_id": 1, "amount": 120.50}',
    '{"order_id": 2, "amount": "N/A"}',
    '{"order_id": 3, "amount": 30.00}',
]))
schema = "order_id int, amount decimal(12,2), _corrupt_record string"

permissive = (
    spark.read.schema(schema)
    .option("mode", "PERMISSIVE").option("columnNameOfCorruptRecord", "_corrupt_record")
    .json(bad_dir).cache()
)
permissive.show(truncate=False)
print("PERMISSIVE   :", permissive.count(), "rows,", permissive.where("_corrupt_record IS NOT NULL").count(), "corrupt")
dropped = spark.read.schema(schema).option("mode", "DROPMALFORMED").json(bad_dir)
print("DROPMALFORMED:", len(dropped.select("order_id", "amount").collect()), "rows when columns are read")
print("DROPMALFORMED:", dropped.count(), "rows from count()")

spark.sparkContext.setLogLevel("OFF")  # the FAILFAST error below is expected
try:
    spark.read.schema(schema).option("mode", "FAILFAST").json(bad_dir).collect()
except Exception as exc:
    reason = next((line for line in str(exc).splitlines() if "MALFORMED" in line.upper()), str(exc).splitlines()[0])
    print("FAILFAST     :", reason.strip()[:90])
finally:
    spark.sparkContext.setLogLevel("ERROR")

**What to notice:** `PERMISSIVE` kept all three rows: order 2 has a null amount and its raw line in `_corrupt_record`, ready to be
quarantined. `DROPMALFORMED` returned two rows when the columns were read, with no hint that one was lost. Yet
`count()` on the same reader returned three: `count()` needs no columns, so Spark skipped parsing and never
found the bad row. Never use `count()` alone to check a malformed-record read. `FAILFAST` stopped the read with
an error.

## Record where each row came from

Every file source exposes a hidden `_metadata` column: a struct with `file_path`, `file_name`, `file_size` and
`file_modification_time`. Selecting it costs nothing extra and lets you trace any row back to its delivery.

**Where it matters**
- **Debugging:** "which file did this bad amount come from?"
- **Reprocessing:** delete and reload exactly the rows from one late or corrected file.

**Watch out for**
- Select the fields you need into normal columns before writing; `_metadata` itself is not written.
- The older `input_file_name()` function still works but is deprecated in favour of `_metadata`.

### Example

Land two delivery files and tag each row with its source file and size.

In [ ]:
deliveries = os.path.join(WORK_DIR, "deliveries")
land("deliveries", "orders_2026-01-05.json", '{"order_id": 1}\n{"order_id": 2}')
land("deliveries", "orders_2026-01-06.json", '{"order_id": 3}')

(
    spark.read.schema("order_id int").json(deliveries)
    .select("order_id", F.col("_metadata.file_name").alias("source_file"), F.col("_metadata.file_size").alias("file_bytes"))
    .orderBy("order_id")
    .show(truncate=False)
)

**What to notice:** Each order carries the name and size of the file it was read from, with no extra read. Stored in bronze, the
`source_file` column lets you answer which delivery a row came from long after the raw files are archived.

## Read partitioned folders

A dataset written with `partitionBy("order_date")` is a folder per value: `order_date=2026-01-05/`,
`order_date=2026-01-06/`, and so on. Reading the top folder turns the folder names back into a column
(**partition discovery**), and a filter on that column skips whole folders (**partition pruning**) without
opening their files.

Other reader options narrow which files are read: `pathGlobFilter` (only names matching a pattern),
`recursiveFileLookup` (read nested folders without partition discovery), and `modifiedAfter` / `modifiedBefore`.

**Where it matters**
- **Daily data:** a query for one day reads one folder out of hundreds.
- **Mixed landing folders:** read only the `*.json` files and ignore marker files.

**Watch out for**
- Partition by a column queries filter on and that has few values, usually a date. Partitioning by `customer_id` creates millions of tiny folders.
- Filtering on an expression of the partition column (for example a string comparison) may still prune, but check the plan.

### Example

Write orders partitioned by date, look at the folders, and read back one day while checking the plan for pruning.

In [ ]:
by_day = os.path.join(WORK_DIR, "orders_by_day")
orders.withColumn("order_date", F.to_date("order_ts")).write.mode("overwrite").partitionBy("order_date").parquet(by_day)
print(sorted(name for name in os.listdir(by_day) if name.startswith("order_date="))[:4], "...")

one_day = spark.read.parquet(by_day).where("order_date = '2026-01-12'")
one_day.select("order_id", "status", "amount", "order_date").show()
print(next(line for line in plan_of(one_day).splitlines() if "PartitionFilters" in line).strip())

**What to notice:** Each date became a folder, and `order_date` came back as a `date` column even though it is not stored inside
the files. `PartitionFilters` shows the date condition was applied to folder names, so only the
`order_date=2026-01-12` folder was read.

## Combine schema versions

Sources change. An **additive** change (a new optional column) is the easy case: older files lack the column.
Two ways to read old and new data together:

- `unionByName(other, allowMissingColumns=True)` when you read each version separately;
- `mergeSchema=True` on a Parquet read, which combines the schemas of all files it reads.

Breaking changes (a column renamed, a type changed from number to text) need an explicit mapping and usually a
quarantine, not a merge.

**Where it matters**
- **A feed adds `currency`** halfway through a month.
- **Long-lived Parquet folders** written by several versions of a job.

**Watch out for**
- `mergeSchema` reads every file's footer, which is slow on large folders; table formats record the schema instead.
- A null in the new column means "not delivered", not "no value". Document which one consumers should assume.

### Example

Write two versions of an order file, the second with a `currency` column, and read them together.

In [ ]:
versions = os.path.join(WORK_DIR, "orders_versions")
orders.select("order_id", "amount").write.mode("overwrite").parquet(os.path.join(versions, "v=1"))
orders.select("order_id", "amount", F.lit("EUR").alias("currency")).write.mode("overwrite").parquet(os.path.join(versions, "v=2"))

print("without mergeSchema:", spark.read.parquet(versions).columns)
merged = spark.read.option("mergeSchema", True).parquet(versions)
print("with mergeSchema:   ", merged.columns)
merged.groupBy("v", "currency").count().orderBy("v").show()

**What to notice:** Without `mergeSchema`, Spark took the schema from one file and `currency` may be missing. With it, all columns
appear, version 1 rows have a null currency, and `v` comes from the folder names.

## Write data: save modes, layout and file count

Every write goes through `df.write` with a **save mode**:

| mode | if the target exists |
| --- | --- |
| `errorifexists` (default) | fail |
| `append` | add files |
| `overwrite` | replace the target (or only the touched partitions, with dynamic overwrite: [lesson 10](pyspark-10-batch-pipelines.ipynb)) |
| `ignore` | do nothing, silently |

Each task writes its own file, so the number of output files is the number of partitions (times the number
of partition folders each task touches). Control it with `repartition(n)` or `coalesce(n)` before writing, or
cap rows per file with `maxRecordsPerFile`. Choose compression with `option("compression", "zstd")`.

**Where it matters**
- **Gold tables** rebuilt on each run: `overwrite`.
- **Append-only logs:** `append`, with deduplication so a retry does not double the data.

**Watch out for**
- `append` in a retried job duplicates data.
- `coalesce(1)` on a large result writes one huge file on one core.
- `overwrite` on a folder deletes everything in it, including partitions your DataFrame does not contain (unless dynamic overwrite is on).

### Example

Write the same data with different partition counts and a row cap, and try each save mode on an existing folder.

In [ ]:
out = os.path.join(WORK_DIR, "layout")
orders.repartition(4).write.mode("overwrite").parquet(out)
print("repartition(4):          ", len(files_in(out, ".parquet")), "files")
orders.coalesce(1).write.mode("overwrite").parquet(out)
print("coalesce(1):             ", len(files_in(out, ".parquet")), "file")
orders.coalesce(1).write.mode("overwrite").option("maxRecordsPerFile", 5).parquet(out)
print("maxRecordsPerFile=5:     ", len(files_in(out, ".parquet")), "files")

orders.write.mode("append").parquet(out)
print("after append:            ", spark.read.parquet(out).count(), "rows")
orders.write.mode("ignore").parquet(out)
print("after ignore:            ", spark.read.parquet(out).count(), "rows")
spark.sparkContext.setLogLevel("OFF")
try:
    orders.write.parquet(out)  # the default mode is errorifexists
except Exception as exc:
    print("errorifexists:           ", str(exc).split("]")[0] + "]")
finally:
    spark.sparkContext.setLogLevel("ERROR")

**What to notice:** Four partitions wrote four files; `coalesce(1)` wrote one; the 5-row cap split that one partition's 12 rows into
three files. `append` doubled the rows to 24, `ignore` changed nothing without saying so, and the default mode
refused to touch an existing path.

## Read from a database over JDBC

`spark.read.format("jdbc")` reads a table or a query from any database with a JDBC driver. By default the whole
read is one query in one task. To read in parallel, give a numeric or date `partitionColumn` with `lowerBound`,
`upperBound` and `numPartitions`; Spark runs one range query per partition. The bounds set the ranges; they do
not filter rows.

```python
orders_db = (
    spark.read.format("jdbc")
    .option("url", "jdbc:postgresql://orders-db.example.internal:5432/shop")
    .option("dbtable", "(select * from orders where updated_at >= now() - interval '1 day') as recent")
    .option("user", user).option("password", password)       # from a secret store, never hard-coded
    .option("partitionColumn", "order_id")
    .option("lowerBound", 1).option("upperBound", 50_000_000)
    .option("numPartitions", 16)                              # 16 parallel range queries
    .option("fetchsize", 10_000)                              # rows per round trip
    .load()
)
```

This course never connects to a live database, so the example is shown, not run. Writing back uses
`df.write.format("jdbc")` with the same options plus a save mode.

**Where it matters**
- **Dimension tables and initial loads** from operational databases.
- **Incremental extracts** with a `WHERE updated_at > :watermark` subquery.

**Watch out for**
- Without `partitionColumn`, a large table is read by one task over one connection.
- `numPartitions` is also the number of concurrent connections to the database. Agree a limit with its owner.
- For large or frequent extracts, change data capture into Kafka or files is kinder to the source than repeated full reads.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** Why give a schema when reading CSV or JSON?

**A.** Without one, CSV is read as strings and JSON is scanned twice to guess types, and the guess can change between deliveries. A schema is faster and enforces the contract.

**Q2.** What does `multiLine` change for JSON?

**A.** It reads a file that is one JSON document or array instead of one object per line. The file is then read by a single task.

**Q3.** Which read mode keeps bad rows for a quarantine?

**A.** `PERMISSIVE` with `columnNameOfCorruptRecord`: bad fields become null and the raw line is kept.

**Q4.** How do you find the file a row came from?

**A.** Select `_metadata.file_name` or `_metadata.file_path` when reading.

**Q5.** What is partition pruning, and how do you check it happened?

**A.** Skipping partition folders whose values cannot match the filter. The plan's `PartitionFilters` shows it.

**Q6.** How many files does a write produce, and how do you control it?

**A.** One per task (times partition folders each task touches). Use `repartition`, `coalesce` or `maxRecordsPerFile`.

## Key takeaways

- Always pass a schema and explicit options for text formats.
- Use Parquet (or a table format) after the landing zone: typed, columnar and prunable.
- Read with `PERMISSIVE` and a corrupt-record column, then reconcile landed = parsed + rejected.
- Keep `_metadata.file_name` in bronze for lineage.
- Partition by a low-cardinality column that queries filter on, usually a date.
- Pick the save mode deliberately; `append` plus a retry means duplicates.

**Next:** Practise this module in [PySpark drills 4: reading and writing data](pyspark-drills-04-reading-and-writing-data.ipynb), then continue with [PySpark 5: Spark SQL and the catalog](pyspark-05-spark-sql-and-the-catalog.ipynb).